In [1]:
# загрузка данных
import pandas as pd

movies = pd.read_csv(
    'ml-1m/movies.dat',
    sep='::',
    engine='python',
    names=['MovieID', 'Title', 'Genres'],
    encoding='latin-1'
)

ratings = pd.read_csv(
    'ml-1m/ratings.dat',
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    encoding='latin-1'
)

users = pd.read_csv(
    'ml-1m/users.dat',
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
    encoding='latin-1'
)

df = ratings.merge(users, on='UserID').merge(movies, on='MovieID')

print(df.head(7))

   UserID  MovieID  Rating  Timestamp Gender  Age  Occupation Zip-code  \
0       1     1193       5  978300760      F    1          10    48067   
1       1      661       3  978302109      F    1          10    48067   
2       1      914       3  978301968      F    1          10    48067   
3       1     3408       4  978300275      F    1          10    48067   
4       1     2355       5  978824291      F    1          10    48067   
5       1     1197       3  978302268      F    1          10    48067   
6       1     1287       5  978302039      F    1          10    48067   

                                    Title                           Genres  
0  One Flew Over the Cuckoo's Nest (1975)                            Drama  
1        James and the Giant Peach (1996)     Animation|Children's|Musical  
2                     My Fair Lady (1964)                  Musical|Romance  
3                  Erin Brockovich (2000)                            Drama  
4                    B

In [2]:
# 1. Средние рейтинги для фильма с ID = № варианта в зависимости от пола
variant_id = 38

average_rating = (
    df[df['MovieID'] == variant_id]
    .groupby('Gender')
    ['Rating'].mean()
    .reset_index()
    .rename(columns={'Rating': 'Mean_Rating'})
)

print(average_rating)

  Gender  Mean_Rating
0      F     3.214286
1      M     2.428571


In [3]:
# 2. Отобрать фильмы, получившие не менее 250 оценок, и для них определить средний рейтинг в зависимости от пола;

ratings_count = (
    df.groupby('MovieID')
    ['Rating'].count()
    .reset_index(name='Ratings_Count')
)

popular_movies = ratings_count[ratings_count['Ratings_Count'] >= 250]

df_popular = df.merge(popular_movies[['MovieID']], on='MovieID')

result = (
    df_popular
    .groupby(['MovieID', 'Title', 'Gender'])
    ['Rating'].mean()
    .unstack('Gender')                    # приведение Gender в колонки
    .reset_index()
    .rename(columns={'F': 'Rating_F', 'M': 'Rating_M'})                     
)

print(result[['MovieID', 'Title', 'Rating_F', 'Rating_M']].to_string(index=False))


 MovieID                                                                       Title  Rating_F  Rating_M
       1                                                            Toy Story (1995)  4.187817  4.130552
       2                                                              Jumanji (1995)  3.278409  3.175238
       3                                                     Grumpier Old Men (1995)  3.073529  2.994152
       5                                          Father of the Bride Part II (1995)  3.212963  2.888298
       6                                                                 Heat (1995)  3.682171  3.909988
       7                                                              Sabrina (1995)  3.588235  3.267717
      10                                                            GoldenEye (1995)  3.470149  3.553050
      11                                              American President, The (1995)  3.923483  3.718654
      16                                               

In [4]:
# 3. Найти фильмы, оказавшиеся в топ 5 у зрителей-женщин и худшие 5 у зрителей- мужчин. Найти фильмы (топ 5), по которым мужчины и женщины сильнее всего разошлись в оценках;

average_rating = (
    df.groupby(['MovieID', 'Title', 'Gender'])
    ['Rating'].mean()
    .unstack('Gender')
    .reset_index()
    .rename(columns={'F': 'Rating_F', 'M': 'Rating_M'})
)

average_rating = average_rating[average_rating['Rating_F'].notna() & average_rating['Rating_M'].notna()]

top_5_women = (
    average_rating
    .sort_values('Rating_F', ascending=False).head(5)
)

worst_5_men = (
    average_rating
    .sort_values('Rating_M', ascending=True).head(5)
)

average_rating['Rating_Diff'] = (average_rating['Rating_F'] - average_rating['Rating_M']).abs()

top_5_diff = (
    average_rating
    .sort_values('Rating_Diff', ascending=False).head(5)
)

print("\nТоп 5 у женщин:\n", top_5_women.to_string(index=False))
print("\nХудшие 5 у мужчин:\n", worst_5_men.to_string(index=False))
print("\nТоп 5 по расхождению:\n", top_5_diff.to_string(index=False))


Топ 5 у женщин:
  MovieID                                             Title  Rating_F  Rating_M
      53                                   Lamerica (1994)       5.0  4.666667
    2675                     Twice Upon a Yesterday (1998)       5.0  3.222222
    3817  Other Side of Sunday, The (Søndagsengler) (1996)       5.0  2.928571
    2538                                 Dancemaker (1998)       5.0  3.250000
     854 Ballad of Narayama, The (Narayama Bushiko) (1958)       5.0  3.428571

Худшие 5 у мужчин:
  MovieID                                                      Title  Rating_F  Rating_M
    3123                Spring Fever USA (a.k.a. Lauderdale) (1989)       1.0       1.0
     834                                          Phat Beach (1996)       3.0       1.0
    1311                                  Santa with Muscles (1996)       1.0       1.0
    3228                                      Wirey Spindell (1999)       1.0       1.0
    3493 Torso (Corpi Presentano Tracce di Viol

In [5]:
# 4. Определить фильмы (топ 5), вызвавшие в среднем наибольшее разногласие у зрителей независимо от пола;

movie_diff = (
    df.groupby(['MovieID', 'Title'])
    ['Rating'].std()  # стандартное отклонение
    .reset_index(name='RatingSpread')
)

top_5_movie_diff = movie_diff.sort_values('RatingSpread', ascending=False).head(5)

print("\nТОП-5 фильмов с наибольшим разногласием зрителей:\n", top_5_movie_diff.to_string(index=False))


ТОП-5 фильмов с наибольшим разногласием зрителей:
  MovieID                                         Title  RatingSpread
     572                        Foreign Student (1994)      2.828427
    3800 Criminal Lovers (Les Amants Criminels) (1999)      2.309401
     706                            Sunset Park (1996)      2.121320
    3828                          Better Living (1998)      2.121320
     887                         Talk of Angels (1998)      2.121320


In [6]:
# 5. Определить средний рейтинг каждого жанра для различных возрастов. Установите, фильмы какого жанра в каждой категории возрастов имеют наибольший и наименьший рейтинги.
# Если фильм относится к нескольким данным, то их надо разделить и сформировать несколько строчек с этим фильмом, но с различными жанрам

df_genres = df.copy()
df_genres['Genres'] = df_genres['Genres'].str.split('|')
df_genres = df_genres.explode('Genres')

display(df_genres)

genre_age_rating = (
    df_genres
    .groupby(['Age', 'Genres'])
    ['Rating'].mean()
    .reset_index()
    .rename(columns={'Rating' : 'Mean_Rating'})
)

print("Средний рейтинг жанров по возрастам:\n")
print(genre_age_rating.head(27).to_string(index=False))

,UserID,MovieID,Rating,Timestamp,Gender,Age,Occupation,Zip-code,Title,Genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Children's
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical
...,...,...,...,...,...,...,...,...,...,...
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Children's
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Drama
1000208,6040,1097,4,956715569,M,25,6,11106,E.T. the Extra-Terrestrial (1982),Fantasy


Средний рейтинг жанров по возрастам:

 Age      Genres  Mean_Rating
   1      Action     3.506385
   1   Adventure     3.449975
   1   Animation     3.476113
   1  Children's     3.241642
   1      Comedy     3.497491
   1       Crime     3.710170
   1 Documentary     3.730769
   1       Drama     3.794735
   1     Fantasy     3.317647
   1   Film-Noir     4.145455
   1      Horror     3.254184
   1     Musical     3.568306
   1     Mystery     3.631522
   1     Romance     3.621284
   1      Sci-Fi     3.478698
   1    Thriller     3.550373
   1         War     3.895437
   1     Western     3.576119
  18      Action     3.447097
  18   Adventure     3.408525
  18   Animation     3.624014
  18  Children's     3.294257
  18      Comedy     3.460417
  18       Crime     3.668054
  18 Documentary     3.865865
  18       Drama     3.721930
  18     Fantasy     3.353778


In [62]:
max_ratings = (
    genre_age_rating.loc[
        genre_age_rating.groupby('Age')['Mean_Rating'].idxmax()
    ]
    .reset_index(drop=True)
)

min_ratings = (
    genre_age_rating.loc[
        genre_age_rating.groupby('Age')['Mean_Rating'].idxmin()
    ]
    .reset_index(drop=True)
)

print("\nЖанры с максимальным рейтингом в каждой возрастной категории:")
print(max_ratings.to_string(index=False))

print("\nЖанры с минимальным рейтингом в каждой возрастной категории:")
print(min_ratings.to_string(index=False))


Жанры с максимальным рейтингом в каждой возрастной категории:
 Age    Genres  Mean_Rating
   1 Film-Noir     4.145455
  18 Film-Noir     3.997368
  25 Film-Noir     4.058725
  35 Film-Noir     4.064910
  45 Film-Noir     4.105376
  50 Film-Noir     4.175401
  56 Film-Noir     4.125932

Жанры с минимальным рейтингом в каждой возрастной категории:
 Age     Genres  Mean_Rating
   1 Children's     3.241642
  18     Horror     3.172682
  25     Horror     3.200096
  35     Horror     3.276022
  45     Horror     3.262274
  50     Horror     3.158940
  56     Horror     3.254401
